In [1]:
! pip install -q langgraph

In [ ]:
import operator
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END

class Brief(TypedDict):
    hours:     int                              # 수집 time window(시간)
    collected: list                             # ① 수집한 기사
    picked:    list                             # ② 선별해 남긴 다섯 건
    drafted:   Annotated[list, operator.add]    # ③ 취재한 초안 — 워커들이 나눠 채운다
    log:       Annotated[list, operator.add]    # 무슨 일이 있었는지

def collect(s: dict) -> dict:                  # ① 자료 수집  — 섹션 4에서 채운다
    return {"collected": [], "log": [f"① 수집   {s['hours']}시간 time window · 0건 (빈 노드)"]}

def select(s: dict) -> dict:                   # ② 중요도 선별 — 섹션 5에서 채운다
    return {"picked": [], "log": [f"② 선별   {len(s['collected'])} → 0건 (빈 노드)"]}

def report(s: dict) -> dict:                   # ③ 요약       — 섹션 6에서 채운다
    return {"drafted": [], "log": [f"③ 취재   {len(s['picked'])}건 (빈 노드)"]}

def verify(s: dict) -> dict:                   # ④ 검수       — 섹션 7에서 채운다
    return {"log": [f"④ 검수   {len(s['drafted'])}건 (빈 노드)"]}

def publish(s: dict) -> dict:                  # ⑤ 발행       — 섹션 8에서 채운다
    return {"log": [f"⑤ 발행   {len(s['drafted'])}건 (빈 노드)"]}

def build():                                    # 노드를 이름으로 찾아 붙인다
    g = StateGraph(Brief)
    for name in ("collect", "select", "report", "verify", "publish"):
        g.add_node(name, globals()[name])
    g.add_edge(START, "collect")
    g.add_edge("collect", "select")
    g.add_edge("select", "report")
    g.add_edge("report", "verify")
    g.add_edge("verify", "publish")
    g.add_edge("publish", END)
    return g

INIT = {"hours": 336,
        "collected": [], "picked": [], "drafted": [], "log": []}

def run():
    return build().compile().invoke(INIT)

for line in run()["log"]:
    print(line)

In [2]:
! pip install -q feedparser requests

In [ ]:
import feedparser, requests, re
from datetime import datetime, timezone

UA = {"User-Agent": "Mozilla/5.0 (newsletter-agent-course)"}
SOURCES = [
    ("I Care If You Listen", "https://icareifyoulisten.com/feed/"),
    ("Sequenza21",           "https://sequenza21.com/feed/"),
    ("VAN Magazine",         "https://van-magazine.com/feed/"),
    ("Guardian Classical",   "https://www.theguardian.com/music/classical-music-and-opera/rss"),
    ("Slipped Disc",         "https://slippedisc.com/feed/"),
]
now = datetime.now(timezone.utc)
print(f"{'소스':<22}{'상태':<6}{'건수':>5}{'요약길이':>9}{'최신글':>9}")
print("-" * 53)
for name, url in SOURCES:
    r = requests.get(url, headers=UA, timeout=20)
    f = feedparser.parse(r.content)
    e = f.entries[0]
    body = e.get("content", [{}])[0].get("value") if e.get("content") else e.get("summary", "")
    text = re.sub(r"<[^>]+>", "", body or "").strip()
    t = e.get("published_parsed")
    age = (now - datetime(*t[:6], tzinfo=timezone.utc)).total_seconds() / 3600 if t else None
    print(f"{name:<22}{r.status_code:<6}{len(f.entries):>5}{len(text):>9}"
          f"{(f'{age:.0f}h' if age is not None else '?'):>9}")

In [ ]:
import feedparser, requests, re
from datetime import datetime, timedelta, timezone

UA = {"User-Agent": "Mozilla/5.0 (newsletter-agent-course)"}
SOURCES = [
    ("I Care If You Listen", "https://icareifyoulisten.com/feed/"),
    ("Sequenza21",           "https://sequenza21.com/feed/"),
    ("VAN Magazine",         "https://van-magazine.com/feed/"),
    ("Guardian Classical",   "https://www.theguardian.com/music/classical-music-and-opera/rss"),
    ("Slipped Disc",         "https://slippedisc.com/feed/"),
]

# 일반 클래식 뉴스라 현대음악 글만 남겨야 하는 소스: 제목·요약에 키워드가 하나라도 있어야 통과
TOPIC_FILTER = {
    "Slipped Disc": re.compile(
        r"contemporary|new music|new work|world premiere|premiere[sd]?\b|composer|commission|"
        r"avant-garde|experimental|minimalis|electroacoustic|ensemble|festival|"
        r"\b(?:20th|21st)[- ]century|living composer",
        re.I),
}

def strip_tags(s):
    return re.sub(r"<[^>]+>", "", s or "").strip()

def published_at(entry):
    t = entry.get("published_parsed")
    return datetime(*t[:6], tzinfo=timezone.utc) if t else None

def collect(s: dict) -> dict:                   # ① 자료 수집 — 빈 노드를 갈아 끼운다
    cutoff = datetime.now(timezone.utc) - timedelta(hours=s["hours"])
    items, dead, empty, seen = [], [], [], set()
    filtered = 0
    for name, url in SOURCES:
        try:
            r = requests.get(url, headers=UA, timeout=20)
            r.raise_for_status()                # 403·404·500도 실패로 잡는다
            feed = feedparser.parse(r.content)
        except Exception:
            dead.append(name)                   # 한 곳이 죽어도 나머지는 계속
            continue
        n_before = len(items)
        for e in feed.entries:
            at = published_at(e)
            if not at or at < cutoff:           # time window 밖이거나 날짜가 없으면 버린다
                continue
            key = e.link.split("?")[0].rstrip("/")       # 추적용 꼬리표를 떼고 비교
            if key in seen:
                continue
            summary = strip_tags(e.get("summary", ""))[:300]
            topic = TOPIC_FILTER.get(name)
            if topic and not topic.search(f"{e.title} {summary}"):
                filtered += 1                   # 주제 밖이라 버린 글
                continue
            seen.add(key)
            items.append({"title": e.title, "url": e.link, "source": name, "at": at,
                          "summary": summary})
        if len(items) == n_before:
            empty.append(name)                  # 응답은 왔지만 window 안에 0건
    return {"collected": items,
            "log": [f"① 수집   {s['hours']}시간 time window · {len(items)}건"
                    + (f" · 주제 필터로 {filtered}건 제외" if filtered else "")
                    + (f" · 응답 없음 {dead}" if dead else "")
                    + (f" · 0건 {empty}" if empty else "")]}

for line in run()["log"]:                       # 그래프는 그대로, 노드만 바뀌었다
    print(line)

In [ ]:
! pip install -q openai feedparser requests
! pip install -q -U "brotli>=1.2.0"

In [ ]:
import os

# 방법 A) .env 파일 사용 (권장)
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

# 방법 B) Colab 보안 비밀 사용
# from google.colab import userdata
# os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# 방법 C) 직접 입력 (노트북 공유 전 반드시 삭제)
# os.environ["OPENAI_API_KEY"] = "sk-..."

os.environ.setdefault("OPENAI_API_KEY", "")
print("키 설정 여부:", "설정됨" if os.environ.get("OPENAI_API_KEY") else "비어 있음 — 위 방법 중 하나로 채워주세요")

키 설정 여부: 설정됨


In [ ]:
import os, feedparser, requests
from collections import Counter
from openai import OpenAI

client = OpenAI(default_headers={"Accept-Encoding": "gzip, deflate"})  # 키는 위 셀에서 설정했습니다
UA = {"User-Agent": "Mozilla/5.0 (newsletter-agent-course)"}

# 4번 셀의 SOURCES를 그대로 쓴다 — 소스를 두 곳에서 따로 관리하면 어긋나기 때문
titles = []
for name, url in SOURCES:
    r = requests.get(url, headers=UA, timeout=20)
    r.raise_for_status()
    for e in feedparser.parse(r.content).entries[:3]:
        titles.append((name, e.title))

RUBRIC = ("당신은 현대음악(Contemporary Classical Music)을 챙겨 보는 독자를 위한 뉴스레터 편집자입니다.\n"
          "아래 기사 제목이 '이번 주 현대음악 흐름을 알려 줄 만한가'라는 "
          "기준에서 얼마나 중요한지 1~10점으로 매기세요. 숫자만 답하세요.")

scores = []
for src, t in titles:                      # ← 기사마다 한 번씩, 서로 모르는 채로
    r = client.chat.completions.create(
        model="gpt-4.1-mini", temperature=0, max_tokens=5,
        messages=[{"role": "system", "content": RUBRIC},
                  {"role": "user", "content": t}])
    scores.append((r.choices[0].message.content.strip(), src, t))

for s, src, t in sorted(scores, key=lambda x: -int(x[0])):
    print(f"{s:<6}{src:<22}{t[:44]}")
dist = Counter(s for s, _, _ in scores)
print("\n점수 분포:", " · ".join(f"{k}점 {v}건" for k, v in sorted(dist.items(), key=lambda kv: -int(kv[0]))))
print(f"서로 다른 점수 값: {len(dist)}개 / 후보 {len(scores)}건")


In [ ]:
# 17장 — 15장에서 따로 점수 매긴 같은 15건을, 이번엔 한 화면에 놓고 한 번에 비교한다
listing = "\n".join(f"{i}. [{s}] {t}" for i, (s, t) in enumerate(titles))
SYS = ("당신은 현대음악(Contemporary Classical Music)을 챙겨 보는 독자를 위한 뉴스레터 편집자입니다.\n"
       "아래 후보를 서로 비교해 '이번 주 현대음악 흐름을 알려 줄 만한' 순서대로 "
       "상위 5건을 고르세요. 번호만 쉼표로 구분해 답하세요. 설명 금지.")

r = client.chat.completions.create(                # ← 호출 한 번, 전부 한 화면에
    model="gpt-4.1-mini", temperature=0, max_tokens=40,
    messages=[{"role": "system", "content": SYS},
              {"role": "user", "content": listing}])
picked = [int(x) for x in r.choices[0].message.content.strip().replace(" ", "").split(",")][:5]

print("한 화면에 놓고 비교한 결과 — 상위 5건")
for rank, i in enumerate(picked, 1):
    print(f"  {rank}위  [{titles[i][0]}] {titles[i][1][:46]}")


In [ ]:
# 19장 — 구조화된 출력: 답의 모양(번호·이유·사건 라벨)을 스키마로 강제한다
from pydantic import BaseModel, Field

class Pick(BaseModel):
    index: int = Field(description="후보 목록에서의 번호")
    title: str = Field(description="고른 후보의 제목을 목록에 적힌 그대로 복사")
    reason: str = Field(description="왜 골랐는지 한국어 한 문장")
    event: str = Field(description="이 기사가 다루는 사건을 짧은 한국어 라벨로. 같은 사건이면 같은 라벨")

class Shortlist(BaseModel):
    picks: list[Pick]

SYS = ("당신은 현대음악(Contemporary Classical Music)을 챙겨 보는 독자를 위한 뉴스레터 편집자입니다.\n"
       "후보를 서로 비교해 '이번 주 현대음악 흐름을 알려 줄 만한' 기사 5건을 고르세요.\n"
       "각 기사에 대해 고른 이유와, 그 기사가 다루는 '사건' 라벨을 함께 답하세요.\n"
       "같은 사건을 다룬 기사에는 반드시 같은 라벨을 붙이세요.")

out = client.chat.completions.parse(            # ← parse: 스키마로 형식을 강제한다
    model="gpt-4.1-mini", temperature=0,
    messages=[{"role": "system", "content": SYS},
              {"role": "user", "content": listing}],
    response_format=Shortlist).choices[0].message.parsed

for p in out.picks:
    print(f"[{p.event:<22}] {titles[p.index][1][:40]}")
    print(f"{'':25}└ {p.reason[:60]}")


In [ ]:
# 20~21장 — select 노드: 묶음마다 예선, 살아남은 것을 한 화면에 놓고 본선 5건, 탈락자는 파일에 남긴다
import json
from collections import Counter
from datetime import datetime

BATCH, TARGET, PASS = 40, 5, 12    # 예선 묶음 크기, 최종 발행 건수, 묶음마다 통과 건수(매체 상한 때문에 넉넉히)
PER_SOURCE = 2                     # 한 매체에서 최종 발행할 수 있는 최대 건수

CRITERIA = ("독자는 현대음악(Contemporary Classical Music)을 챙겨 듣고 연주·작곡·기획하는 사람입니다.\n"
            "- 새 작품의 초연·위촉·신보 발매, 생존 작곡가·앙상블의 활동인가\n"
            "- 현대음악 페스티벌·기획, 분야의 흐름이나 논쟁(창작 환경·지원·기술)을 다루는가\n"
            "버릴 것: 바흐~말러 같은 정전 레퍼토리 중심 공연 리뷰, 현대음악과 무관한 부고·인사·스캔들, "
            "시즌·티켓 홍보성 소식")

def ask_picks(items, n):
    listing = "\n".join(f"{i}. [{it['source']}] {it['title']}" for i, it in enumerate(items))
    sys = (f"{CRITERIA}\n\n아래 목록에서 중요한 순서대로 {n}건을 고르세요.\n"
           "reason은 한국어 한 문장으로 쓰세요.\n"
           "같은 사건을 다룬 기사에는 같은 event 라벨을 붙이세요.")
    out = client.chat.completions.parse(
        model="gpt-4.1-mini", temperature=0,
        messages=[{"role": "system", "content": sys},
                  {"role": "user", "content": listing}],
        response_format=Shortlist).choices[0].message.parsed
    by_title = {it["title"]: i for i, it in enumerate(items)}
    seen, picks = set(), []
    for p in out.picks:                          # 없는 번호·겹친 번호는 버린다
        p.index = by_title.get(p.title.strip(), p.index)     # 제목이 일치하면 제목을 믿는다(번호는 가끔 어긋난다)
        if 0 <= p.index < len(items) and p.index not in seen:
            seen.add(p.index)
            picks.append(p)
    return picks

def select(s: dict) -> dict:                    # ② 중요도 선별 — 빈 노드를 갈아 끼운다
    items = s["collected"]
    if not items:
        return {"picked": [], "log": ["② 선별   0 → 0건"]}
    survivors, gates = [], []
    for i in range(0, len(items), BATCH):        # 예선 — 묶음마다 여덟 건
        chunk = items[i:i + BATCH]
        ranked = ask_picks(chunk, PASS + 1)      # 9등까지 받아 통과선(8등)과 첫 탈락(9등)을 남긴다
        survivors += [chunk[p.index] for p in ranked[:PASS]]
        gates.append({"pass_last": chunk[ranked[PASS - 1].index]["title"] if len(ranked) >= PASS else None,
                      "first_out": chunk[ranked[PASS].index]["title"] if len(ranked) > PASS else None})
    ranked = ask_picks(survivors, len(survivors))  # 본선 — 한 화면에 놓고 전부 순위를 받아 둔다(건너뛴 자리를 채울 예비)
    seen_events, per_src, finals, dropped = set(), Counter(), [], []
    for p in ranked:
        label = p.event.strip().lower()
        src = survivors[p.index]["source"]
        if len(finals) >= TARGET:
            dropped.append((p, "순위 밖"))
        elif label in seen_events:                  # 같은 사건은 한 건만
            dropped.append((p, "같은 사건"))
        elif per_src[src] >= PER_SOURCE:            # 한 매체가 자리를 다 차지하지 못하게
            dropped.append((p, "매체 상한"))
        else:
            seen_events.add(label)
            per_src[src] += 1
            finals.append(p)
    relaxed = 0
    for item in list(dropped):                      # 상한 때문에 5건이 안 되면, 상한에 밀린 것부터 순위대로 채운다
        if len(finals) >= TARGET:
            break
        if item[1] == "매체 상한":
            dropped.remove(item)
            finals.append(item[0])
            relaxed += 1
    finals.sort(key=ranked.index)
    picked = [{**survivors[p.index], "event": p.event, "reason": p.reason} for p in finals]
    ratio = len(finals) / len(survivors)
    rec = {"at": datetime.now().isoformat(timespec="seconds"), "hours": s["hours"],
           "collected": len(items), "survivors": len(survivors), "ratio": round(ratio, 2),
           "gates": gates,
           "picked": [{"title": x["title"], "event": x["event"], "reason": x["reason"]} for x in picked],
           "dropped": [{"title": survivors[p.index]["title"], "event": p.event, "reason": p.reason,
                        "why": why} for p, why in dropped]}
    with open("select_log.jsonl", "a", encoding="utf-8") as f:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    return {"picked": picked,
            "log": [f"② 선별   {len(items)} → 예선 {len(survivors)} → {len(finals)}건 · 발행 비율 {ratio:.2f}"
                    + (f" · 매체 상한 완화 {relaxed}건" if relaxed else "")
                    + (f" · 5건 미달({len(finals)}건)" if len(finals) < TARGET else "")]}

result = run()
for line in result["log"]:
    print(line)
print()
for it in result["picked"]:
    print(f"[{it['event']:<20}] [{it['source']}] {it['title'][:50]}")
    print(f"{'':23}└ {it['reason']}")


In [ ]:
! pip install -q trafilatura


In [ ]:
# 22~24장 — 취재(report)의 요약은 headline · summary · why 세 칸으로 받는다
import re, trafilatura
from pydantic import BaseModel, Field

class Draft(BaseModel):
    headline: str = Field(description="한국어 헤드라인. 30자 이내, 마침표 없음")
    summary:  str = Field(description="정확히 세 문장. 문장마다 ~습니다/~합니다로 끝나는 합니다체. 과장 없이 건조하게")
    why:      str = Field(description="현대음악을 챙겨 보는 독자에게 왜 중요한지 정확히 한 문장, 80자 이내")

SYS_REPORT = ("당신은 현대음악(Contemporary Classical Music) 뉴스레터 기자입니다.\n"
              "아래 기사 본문을 읽고 헤드라인·요약·왜 중요한지를 한국어로 쓰세요.\n"
              "본문에 없는 사실·숫자·평가는 쓰지 마세요.\n"
              "'주목된다·기대를 모은다' 같은 기자체 표현은 쓰지 마세요.\n"
              "형식 규칙(반드시 지키세요):\n"
              "- headline: 30자 이내, 마침표 없음\n"
              "- summary: 정확히 세 문장, 모든 문장이 '~습니다/~합니다'로 끝남(~했다·~이다 금지)\n"
              "- why: 정확히 한 문장, 80자 이내\n"
              "- 작품명·인명은 원문 표기 그대로 두세요")

# 24장 — 프롬프트는 부탁이지 보장이 아니다. 출력만 보고 확인할 수 있는 규칙은 코드로 검사한다
SENT_END = re.compile(r"(?:다|요|죠)[.!?]['’”\"]?(?=\s|$)")      # 문장 끝(No. 2 같은 약어의 점은 세지 않는다)
FORMAL_END = re.compile(r"(?<!니)다[.!?]['’”\"]?(?=\s|$)")        # ~했다·~이다처럼 합니다체가 아닌 끝

def check_draft(d):
    """규칙을 어긴 항목 목록. 비어 있으면 통과."""
    issues = []
    if not re.search(r"[가-힣]", d.headline + d.summary):
        issues.append("한국어가 아님")
    if len(d.headline) > 40:                                   # 30자 이내 요청, 10자는 여유(영어 인명이 길다)
        issues.append(f"headline {len(d.headline)}자(30자 이내여야 함)")
    if len(SENT_END.findall(d.summary)) != 3:
        issues.append(f"summary {len(SENT_END.findall(d.summary))}문장(정확히 3문장이어야 함)")
    if FORMAL_END.search(d.summary):
        issues.append("summary가 합니다체가 아님(~했다·~이다 금지)")
    if len(SENT_END.findall(d.why)) != 1 or len(d.why) > 90:    # 80자 이내 요청, 10자는 여유
        issues.append(f"why {len(SENT_END.findall(d.why))}문장·{len(d.why)}자(한 문장, 80자 이내여야 함)")
    return issues

for it in result["picked"][:3]:                      # 10번 셀의 선별 결과
    downloaded = trafilatura.fetch_url(it["url"])
    body = trafilatura.extract(downloaded) if downloaded else None
    if not body or len(body) < 600:                  # ← 11장에서 정한 G1 기준선
        continue
    out = client.chat.completions.parse(
        model="gpt-4.1-mini", temperature=0,
        messages=[{"role": "system", "content": SYS_REPORT},
                  {"role": "user", "content": body[:6000]}],
        response_format=Draft).choices[0].message.parsed
    print("─" * 66)
    print("원문:", it["title"][:60], f"(본문 {len(body)}자)")
    print("  헤드라인:", out.headline)
    print("  요약    :", out.summary[:120], "…")
    print("  규칙 위반:", check_draft(out) or "없음")


In [ ]:
# 25장 — report 노드: 기사 수만큼 Send로 펼친다 (fan-out / fan-in)
from langgraph.types import Send

class ReportIn(TypedDict):                           # 워커가 받는 것은 기사 하나뿐
    item: dict

def extract_body(url):
    d = trafilatura.fetch_url(url)
    return trafilatura.extract(d) if d else None

MAX_TRY = 3

def draft(body):
    """형식 검사를 통과할 때까지 최대 3번. 반환: (초안 또는 None, 마지막 위반 목록)"""
    sys_msg, issues = SYS_REPORT, []
    for k in range(MAX_TRY):
        d = client.chat.completions.parse(
            model="gpt-4.1-mini", temperature=0 if k == 0 else 0.4,    # 재시도는 같은 답이 반복되지 않게 온도를 올린다
            messages=[{"role": "system", "content": sys_msg},
                      {"role": "user", "content": body[:6000]}],
            response_format=Draft).choices[0].message.parsed
        issues = check_draft(d)
        if not issues:
            return d, []
        sys_msg = SYS_REPORT + "\n\n앞선 답이 규칙을 어겼습니다: " + "; ".join(issues) + ". 고쳐서 다시 쓰세요."
    if len(issues) == 1 and issues[0].startswith("headline"):      # 헤드라인만 길면 그 칸만 따로 줄인다
        for _ in range(2):
            d.headline = client.chat.completions.create(
                model="gpt-4.1-mini", temperature=0, max_tokens=60,
                messages=[{"role": "system", "content": "한국어 헤드라인을 뜻을 유지하며 25자 이내로 줄이세요. "
                                                        "인명·작품명은 줄여도 됩니다. 헤드라인만 답하세요."},
                          {"role": "user", "content": d.headline}]).choices[0].message.content.strip().rstrip(".")
            if not check_draft(d):
                return d, []
    return None, issues

def fan_report(s: dict):                             # 기사 수만큼 워커를 펼친다
    return [Send("report", {"item": it}) for it in s["picked"]] or "verify"   # 0건인 날에도 verify로 간다

def report(s: ReportIn) -> dict:                     # ③ 요약 — 기사 한 건을 맡는다
    it = s["item"]
    body = extract_body(it["url"])
    if not body or len(body) < 600:                  # ← 11장에서 정한 G1 기준선
        return {"drafted": [],
                "log": [f"   취재 제외 {it['source']} · 본문 {len(body or '')}자"]}
    d, issues = draft(body)
    if d is None:                                    # 세 번 써도 규칙을 못 지키면 빠진 이름을 남긴다
        return {"drafted": [], "log": [f"   취재 제외 {it['source']} · 요약 형식 위반 {issues}"]}
    return {"drafted": [{**it, "body": body[:6000], **d.model_dump()}]}

def build():                                         # 엣지 한 줄이 팬아웃으로 바뀐다
    g = StateGraph(Brief)
    for name in ("collect", "select", "report", "verify", "publish"):
        g.add_node(name, globals()[name])
    g.add_edge(START, "collect")
    g.add_edge("collect", "select")
    g.add_conditional_edges("select", fan_report, ["report", "verify"])   # ← 고정 엣지가 아니라
    g.add_edge("report", "verify")
    g.add_edge("verify", "publish")
    g.add_edge("publish", END)
    return g

result = run()
for line in result["log"]:
    print(line)
print()
for it in sorted(result["drafted"], key=lambda x: x["source"]):
    print(f"[{it['source']}] {it['headline']}")
    print(f"  요약: {it['summary']}")
    print(f"  왜  : {it['why']}\n")


In [ ]:
# 27장 — 가장 먼저 떠오르는 검사: 요약의 숫자가 원문에 있는지 글자로 대조한다 (비용 0)
for it in result["drafted"]:                         # 13번 셀의 취재 결과
    nums = sorted(set(re.findall(r"\d[\d,\.]*", it["headline"] + " " + it["summary"])))
    missing = [n for n in nums if n.replace(",", "") not in it["body"].replace(",", "")]
    print(f"[{it['source'][:12]:<12}] 요약의 숫자 {nums} · 원문에 없는 숫자 {missing}")
print("\n→ 영어 원문과 한국어 요약은 글자가 달라(three ≠ 3, 90th ≠ 90번째) 멀쩡한 요약도 걸린다. 반대로 5가 50이 돼도 통과한다.")


In [ ]:
# 28~31장 — verify 노드: LLM 대조(요약을 쓴 호출과 따로) + 연도 글자 대조, 불합격은 버리고 로그에 남기고, 통과분은 새 칸 verified에 담는다
class Verdict(BaseModel):
    ok:       bool      = Field(description="요약이 원문에 근거하면 true")
    problems: list[str] = Field(description="원문에 근거가 없거나 원문과 다른 부분. 없으면 빈 목록")

SYS_CHECK = ("요약이 원문에서 뒷받침되는지 판정하세요.\n"
             "헤드라인과 요약만 보고 판단하고, 번역이나 단위 환산은 문제가 아닙니다.\n"
             "원문에 없는 연도·숫자·사실, 원문과 다르게 옮긴 부분이 있으면 problems에 한 줄씩 적고 ok를 false로 하세요.")

def year_check(d):
    """요약의 4자리 연도가 원문에 없으면 그 목록. 연도는 영어·한국어 모두 아라비아 숫자라 번역 오탐이 없다(LLM 판정이 놓친 연도 환각용)."""
    years = re.findall(r"(?<!\d)(?:19|20)\d{2}(?!\d)", d["headline"] + " " + d["summary"])
    return [y for y in years if y not in d["body"]]

def check(d):                                        # 요약을 쓴 호출과 따로 부른다
    user = (f"[원문]\n{d['body'][:6000]}\n\n"
            f"[헤드라인]\n{d['headline']}\n\n[요약]\n{d['summary']}")
    return client.chat.completions.parse(
        model="gpt-4.1-mini", temperature=0,
        messages=[{"role": "system", "content": SYS_CHECK},
                  {"role": "user", "content": user}],
        response_format=Verdict).choices[0].message.parsed

class Brief(TypedDict):                              # State에 칸을 하나 더한다 (31장)
    hours:     int
    collected: list
    picked:    list
    drafted:   Annotated[list, operator.add]
    verified:  list                                  # ← 검수를 통과한 것(줄이는 칸이라 리듀서 없음)
    log:       Annotated[list, operator.add]

INIT = {**INIT, "verified": []}

def verify(s: dict) -> dict:                         # ④ 검수 — 빈 노드를 갈아 끼운다
    kept, notes = [], []
    for d in s["drafted"]:
        bad_years = year_check(d)                    # 비용 0인 검사를 먼저
        if bad_years:
            notes.append(f"   불합격 {d['source']} · 원문에 없는 연도 {bad_years}")
            continue
        try:
            v = check(d)
        except Exception as e:                       # 판정이 실패한 기사는 보내지 않고 이름을 남긴다
            notes.append(f"   불합격 {d['source']} · 검수 호출 실패({type(e).__name__})")
            continue
        if v.ok:
            kept.append(d)
        else:
            notes.append(f"   불합격 {d['source']} · {' / '.join(v.problems)[:120]}")
    return {"verified": kept,
            "log": [f"④ 검수   {len(s['drafted'])} → {len(kept)}건 · 불합격 {len(s['drafted']) - len(kept)}건"] + notes}

final = run()
for line in final["log"]:
    print(line)


In [ ]:
# 29장 — 검수가 정말 일하는지는 일부러 틀린 요약을 넣어 본다 (조용한 실패 방지)
import re, copy

def num_check(d):                                    # 27장의 글자 대조
    nums = re.findall(r"\d[\d,\.]*", d["headline"] + " " + d["summary"])
    return not [n for n in nums if n.replace(",", "") not in d["body"].replace(",", "")]

base = next((d for d in result["drafted"] if re.search(r"\d", d["summary"])), result["drafted"][0])
fake_num = copy.deepcopy(base)                       # ① 숫자를 하나 바꾼 가짜
m = re.search(r"\d+", fake_num["summary"])
fake_num["summary"] = fake_num["summary"][:m.start()] + str(int(m.group()) * 5 + 7) + fake_num["summary"][m.end():]
fake_year = copy.deepcopy(base)                      # ③ 원문에 없는 연도를 끼운 가짜
fake_year["summary"] = "2023년 10월에 열린 공연입니다. " + fake_year["summary"]
fake_claim = copy.deepcopy(base)                     # ② 숫자 없는 거짓말을 한 문장 끼운 가짜
fake_claim["summary"] += " 이 공연은 현대음악 역사상 처음으로 열린 행사입니다."

print(f"기준 기사: [{base['source']}] {base['headline']}\n")
print(f"{'':<18}{'숫자 대조':<10}{'연도 대조':<10}{'LLM 대조'}")
for name, d in (("진짜 요약", base), ("숫자를 바꾼 가짜", fake_num), ("연도를 끼운 가짜", fake_year),
                ("거짓말을 끼운 가짜", fake_claim)):
    v = check(d)
    print(f"{name:<18}{'합격' if num_check(d) else '불합격':<10}{'불합격' if year_check(d) else '합격':<10}"
          f"{'합격' if v.ok else '불합격'}  {v.problems[:1] if not v.ok else ''}")
print("\n→ 정상이라면 진짜는 모두 합격, 가짜 셋은 연도 대조나 LLM 대조 중 하나에서 불합격이어야 한다. 가짜가 통과하면 검수가 안 돌고 있는 것.")


In [ ]:
# 34장 — embed 카드에 세 칸(headline·summary·why)을 나눠 담고, Discord 한도 안으로 자른다
import json, os, requests

DEFAULT = 0x5F7476
TITLE_MAX, DESC_MAX, EMBED_MAX, TOTAL_MAX = 256, 4096, 10, 5800     # 합계 6000에서 여유를 둔다

def build_embeds(run_id, lead, articles):
    if not articles:                                   # 조용한 날에도 한 장은 보낸다
        return [{"title": f"🗞️ {run_id}", "color": DEFAULT, "description": "오늘은 조용합니다."}]
    embeds = [{"title": f"🗞️ {run_id} · 현대음악 브리핑", "description": lead, "color": DEFAULT}]
    for i, a in enumerate(articles, 1):
        desc = a["summary"]
        if a.get("why"):
            desc += f"\n\n💡 **{a['why']}**"
        embeds.append({
            "title":       f"{i}. {a['headline']}"[:TITLE_MAX],
            "description": desc[:DESC_MAX],
            "url":         a["url"],
            "color":       DEFAULT,
            "footer":      {"text": f"{a['source']} · {a['when']}"},
        })
    total = lambda es: sum(len(e.get("title", "")) + len(e.get("description", ""))
                           + len(e.get("footer", {}).get("text", "")) for e in es)
    while len(embeds) > EMBED_MAX or total(embeds) > TOTAL_MAX:
        embeds.pop()                                   # 뒤에서부터 덜어낸다
    return embeds

def send(run_id, lead, articles, webhook=None, dry_run=True):
    payload = {"username": "편집실", "embeds": build_embeds(run_id, lead, articles)}
    if dry_run or not webhook:                         # 웹훅이 없어도 보내지 않는다
        print(f"[dry-run] embed {len(payload['embeds'])}개 · "
              f"{len(json.dumps(payload, ensure_ascii=False))}자 — 보내지 않음")
        print(json.dumps(payload["embeds"][-1], ensure_ascii=False, indent=2)[:400])
        return False
    r = requests.post(webhook, json=payload, timeout=20)
    ok = r.status_code in (200, 204)
    print("발행:", "성공" if ok else f"실패 {r.status_code} {r.text[:120]}")
    return ok

# 가짜 기사 두 건으로 시험 · 항상 dry_run=True
ARTICLES = [
    {"headline": "가짜 기사 하나", "url": "https://example.com/a", "summary": "첫 번째 시험용 요약입니다. 두 번째 문장입니다. 세 번째 문장입니다.",
     "why": "시험용 이유입니다.", "source": "Sequenza21", "when": "10-27 09:00"},
    {"headline": "가짜 기사 둘", "url": "https://example.com/b", "summary": "또 다른 시험용 요약입니다. 두 번째 문장입니다. 세 번째 문장입니다.",
     "why": "또 다른 시험용 이유입니다.", "source": "I Care If You Listen", "when": "10-27 10:00"},
]
send("2026-10-29", "시험용 머리말입니다.", ARTICLES, webhook=None, dry_run=True)


In [ ]:
# 35장 — publish 노드: 검수를 통과한 verified만 발행한다. 기본은 '보내지 않음'(dry_run)
def make_lead(arts):
    if not arts:
        return ""
    srcs = ", ".join(dict.fromkeys(a["source"] for a in arts))
    return f"오늘은 {len(arts)}건을 골랐습니다. ({srcs})"

def publish(s: dict) -> dict:                          # ⑤ 발행 — 마지막 빈 노드를 갈아 끼운다
    arts = [{"headline": a["headline"], "summary": a["summary"], "why": a["why"],
             "url": a["url"], "source": a["source"],
             "when": a["at"].strftime("%m-%d %H:%M")} for a in s["verified"]]   # 필요한 칸만 새로 만든다(body 제외)
    today = datetime.now().strftime("%Y-%m-%d")
    sent = send(today, make_lead(arts), arts,
                webhook=os.environ.get("DISCORD_WEBHOOK_URL"),
                dry_run=os.environ.get("DRY_RUN", "1") == "1")                  # 기본은 보내지 않음
    label = f"{len(arts)}건" if arts else "조용합니다"
    return {"log": [f"⑤ 발행   {label} · {'보냄' if sent else 'dry-run'}"]}

final = run()
for line in final["log"]:
    print(line)
print()
for i, a in enumerate(final["verified"], 1):          # 발행될 브리핑
    print(f"{i}. {a['headline']}  [{a['source']}]")
    print(f"   {a['summary']}")
    print(f"   💡 {a['why']}")
    print(f"   {a['url']}\n")
